In [1]:
import pandas as pd
import numpy as np
from pybedtools import BedTool
from scipy.stats import hypergeom
from statsmodels.stats.multitest import multipletests
import seaborn as sns
import matplotlib.pyplot as plt

In [2]:
# LQ / Longevity -- Pooled associated OCR with LQ
DIR1='/Aging/LQ_TACIT/phylolm/'
lq_pos=pd.read_csv(DIR1+'phylolm_r0_s1_bh_corrected_pos.bed', sep='\t', header=None, names=['chr', 'start', 'end','peak'])
lq_neg=pd.read_csv(DIR1+'phylolm_r0_s1_bh_corrected_neg.bed', sep='\t', header=None, names=['chr', 'start', 'end','peak'])

# Aging Clock -- Differential OCR in vulnerable AD Inhibitory neurons (negative intrinsic age)
DIR2='/Aging/Aging_SNAIL/data/threshold_90_10/'
aging_pos=pd.read_csv(DIR2+'positives/Anderson_PFC_AD_Inh_peaks_neg24_vs_all_hg38_positive.bed', sep='\t', header=None, names=['chr', 'start', 'end','peak'])
aging_neg=pd.read_csv(DIR2+'negatives/Anderson_PFC_AD_Inh_peaks_neg24_vs_all_hg38_negative.bed', sep='\t', header=None, names=['chr', 'start', 'end','peak'])

In [3]:
# Create background -- merging aging + longevity peaks 

# Combine all peaks
all_peaks = pd.concat([lq_pos, lq_neg, aging_pos, aging_neg], ignore_index=True)
all_peaks = all_peaks.drop_duplicates(subset=['chr','start','end']).reset_index(drop=True)

lq_peaks = pd.concat([lq_pos, lq_neg], ignore_index=True)
lq_peaks = lq_peaks.drop_duplicates(subset=['chr','start','end']).reset_index(drop=True)

aging_peaks = pd.concat([aging_pos, aging_neg], ignore_index=True)
aging_peaks = aging_peaks.drop_duplicates(subset=['chr','start','end']).reset_index(drop=True)

# Background (aging_pos + lq_neg) for GREAT
bgd = pd.concat([lq_neg,aging_pos], ignore_index=True)
bgd = bgd.drop_duplicates(subset=['chr','start','end','peak']).reset_index(drop=True)

In [23]:
# -----------------------------
# 1. Load LQ data pooled + per cell type
# -----------------------------

# Pooled 

# Remove overlapping peaks between pos and neg sets 
pos_bed = BedTool.from_dataframe(lq_pos)
neg_bed = BedTool.from_dataframe(lq_neg)
pos = pos_bed.intersect(neg_bed, v=True)
neg = neg_bed.intersect(pos_bed, v=True)

lq_beds = {}
lq_beds["all_nuclei_pos"] = pos
lq_beds["all_nuclei_neg"] = neg

# Per cell type
celltypes = ['L2.3.IT','L4.5.IT','L5.6.IT.Car3','L5.6.NP','L5.ET','L6.CT','L6.IT',
             'LAMP5','PAX6','PVALB','SST','VIP','Astro','Microglia','Oligo','Opc','Endo']
for ct in celltypes:
    #print(ct)
    pos_bed = pd.read_csv(
        f"{DIR1}{ct}/phylolm_r0_s1_bh_corrected_pos_0.05.bed",
        sep="\t", header=None, names=['chr','start','end','peak'])
    neg_bed = pd.read_csv(
        f"{DIR1}{ct}/phylolm_r0_s1_bh_corrected_neg_0.05.bed",
        sep="\t", header=None, names=['chr','start','end','peak'])

    # Remove overlapping peaks between pos and neg sets 
    pos_bed = BedTool.from_dataframe(pos_bed)
    neg_bed = BedTool.from_dataframe(neg_bed)
    pos = pos_bed.intersect(neg_bed, v=True)
    neg = neg_bed.intersect(pos_bed, v=True)

    # Convert to BedTool
    lq_beds[f"{ct}_pos"] = pos
    lq_beds[f"{ct}_neg"] = neg

# -----------------------------
# 2. Load Aging data
# -----------------------------
aging_beds = {
    "aging_pos": BedTool.from_dataframe(aging_pos),
    "aging_neg": BedTool.from_dataframe(aging_neg),
}

# -----------------------------
# 3. Compute overlaps
# -----------------------------
results = []
for lq_name, lq_bed in lq_beds.items():
    for aging_name, aging_bed in aging_beds.items():
        intersect = lq_bed.intersect(aging_bed, u=True)
        count = len(intersect)
        results.append({
            "aging_set": aging_name,
            "lq_set": lq_name,
            "n_intersections": count
        })

# -----------------------------
# 4. Create tables
# -----------------------------
df = pd.DataFrame(results)
count_table = df.pivot(index="aging_set", columns="lq_set", values="n_intersections")

In [24]:
# Write Signficant Peaks bed files

# Pooled 
lq_beds = {
    "all_nuclei_pos": lq_pos,
    "all_nuclei_neg": lq_neg,
}
# Per cell type
celltypes = ['L2.3.IT','L4.5.IT','L5.6.IT.Car3','L5.6.NP','L5.ET','L6.CT','L6.IT',
             'LAMP5','PAX6','PVALB','SST','VIP','Astro','Microglia','Oligo','Opc','Endo']
for ct in celltypes:
    #print(ct)
    pos = pd.read_csv(
        f"{DIR1}{ct}/phylolm_r0_s1_bh_corrected_pos_0.05.bed",
        sep="\t", header=None, names=['chr','start','end','peak'])
    neg = pd.read_csv(
        f"{DIR1}{ct}/phylolm_r0_s1_bh_corrected_neg_0.05.bed",
        sep="\t", header=None, names=['chr','start','end','peak'])
    
    # Convert to BedTool
    lq_beds[f"{ct}_pos"] = pos
    lq_beds[f"{ct}_neg"] = neg

In [25]:
# Write Overlap bed files

# Foreground 

# Pooled 
# Remove overlapping peaks between pos and neg sets 
pos_bed = BedTool.from_dataframe(lq_pos)
neg_bed = BedTool.from_dataframe(lq_neg)
pos = pos_bed.intersect(neg_bed, v=True)
neg = neg_bed.intersect(pos_bed, v=True)

lq_beds = {}
lq_beds["all_nuclei_pos"] = pos
lq_beds["all_nuclei_neg"] = neg

# Per cell type
celltypes = ['L2.3.IT','L4.5.IT','L5.6.IT.Car3','L5.6.NP','L5.ET','L6.CT','L6.IT',
             'LAMP5','PAX6','PVALB','SST','VIP','Astro','Microglia','Oligo','Opc','Endo']
for ct in celltypes:
    #print(ct)
    pos_bed = pd.read_csv(
        f"{DIR1}{ct}/phylolm_r0_s1_bh_corrected_pos_0.05.bed",
        sep="\t", header=None, names=['chr','start','end','peak'])
    neg_bed = pd.read_csv(
        f"{DIR1}{ct}/phylolm_r0_s1_bh_corrected_neg_0.05.bed",
        sep="\t", header=None, names=['chr','start','end','peak'])

    # Remove overlapping peaks between pos and neg sets 
    pos_bed = BedTool.from_dataframe(pos_bed)
    neg_bed = BedTool.from_dataframe(neg_bed)
    pos = pos_bed.intersect(neg_bed, v=True)
    neg = neg_bed.intersect(pos_bed, v=True)

    # Convert to BedTool
    lq_beds[f"{ct}_pos"] = pos
    lq_beds[f"{ct}_neg"] = neg


overlap_beds = {}
for lq_name, lq_bed in lq_beds.items():
    for aging_name, aging_bed in aging_beds.items():
        intersect = lq_bed.intersect(aging_bed, u=True)
        overlap_beds[f"{lq_name}_{aging_name}"] = intersect.to_dataframe(names=['chr', 'start', 'end', 'peak'])

# Background (aging_pos + lq_neg)
bgd = pd.concat([lq_neg,aging_pos], ignore_index=True)
bgd = bgd.drop_duplicates(subset=['chr','start','end','peak']).reset_index(drop=True)

In [28]:
DIR='/Aging/LQ_TACIT/data/tables/'

#bgd.to_csv(DIR+'bgd_all_nuclei_longevity_neg_aging_pos.bed',index = False, sep ='\t', header = None)
#overlap_beds['all_nuclei_neg_aging_pos'].to_csv(DIR+'Overlap_all_nuclei_neg_aging_pos.bed',index = False, sep ='\t', header = None)
#overlap_beds['PVALB_neg_aging_pos'].to_csv(DIR+'Overlap_PVALB_neg_aging_pos.bed',index = False, sep ='\t', header = None)
#overlap_beds['SST_neg_aging_pos'].to_csv(DIR+'Overlap_SST_neg_aging_pos.bed',index = False, sep ='\t', header = None)
overlap_beds['L5.ET_neg_aging_pos'].to_csv(DIR+'Overlap_L5.ET_neg_aging_pos.bed',index = False, sep ='\t', header = None)


In [9]:
# Write peak tables 

# LQ / Longevity -- Pooled associated OCR with LQ

lq_pos.to_csv(DIR+'phylolm_r0_s1_bh_corrected_pos.csv', index=False)
lq_neg.to_csv(DIR+'phylolm_r0_s1_bh_corrected_neg.csv', index=False)

# Aging Clock -- Differential OCR in vulnerable AD Inhibitory neurons (negative intrinsic age)
aging_pos.to_csv(DIR+'Anderson_PFC_AD_Inh_peaks_neg24_vs_all_hg38_positive_open.csv', index=False, )
aging_neg.to_csv(DIR+'Anderson_PFC_AD_Inh_peaks_neg24_vs_all_hg38_negative_closed.csv', index=False)

all_nuclei_neg_aging_pos = pd.read_csv(DIR+'Overlap_all_nuclei_neg_aging_pos.bed', 
                                       sep ='\t', header = None, names=['chr', 'start', 'end','peak'])
all_nuclei_neg_aging_pos.to_csv(
    DIR+'Overlap_all_nuclei_neg_aging_pos.csv',
    index=False
)

In [ ]:
# Generate random peak sets -- 427 peaks

# Random 1
# Generate bed file with random peak sets
#lq_neg = lq_neg[~lq_neg['peak'].isin(all_nuclei_neg_aging_pos['peak'])]
#lq_sample = lq_neg.sample(n=427, random_state=42)

#lq_sample.to_csv(DIR+'all_nuclei_lq_neg_427_random.bed', index = False, sep ='\t', header = None)
lq_sample=pd.read_csv(DIR+'all_nuclei_lq_neg_427_random.bed',sep ='\t',header = None, names=['chr', 'start', 'end','peak'])

# Random 2
# Generate bed file with random peak sets
lq_neg = lq_neg[~lq_neg['peak'].isin(all_nuclei_neg_aging_pos['peak'])]
lq_neg = lq_neg[~lq_neg['peak'].isin(lq_sample['peak'])]
lq_sample2 = lq_neg.sample(n=427, random_state=0)
len(np.intersect1d(lq_sample['peak'], lq_sample2['peak']))

lq_sample2.to_csv(DIR+'all_nuclei_lq_neg_427_random2.bed', index = False, sep ='\t', header = None)
#lq_sample2=pd.read_csv(DIR+'all_nuclei_lq_neg_427_random.bed',sep ='\t',header = None, names=['chr', 'start', 'end','peak'])

In [116]:
# -----------------------------
# Enrichment Analysis 
# -----------------------------

#M = universe of all unique peaks (background)
#n = number of peaks in Aging set (sample size)
#k = number of peaks in LQ set (successes in background)
#x = observed number of overlaps between Aging and LQ

#M = len(all_peaks)  # total number of unique peaks in your background
M = 627480
pvals = pd.DataFrame(index=count_table.index, columns=count_table.columns)
for aging in count_table.index:
    for lq in count_table.columns:
        x = count_table.loc[aging, lq]   # observed overlaps
        n = len(aging_beds[aging])           # total Aging peaks
        k = len(lq_beds[lq])                 # total LQ peaks
        # P(X >= x)
        pvals.loc[aging, lq] = hypergeom.sf(x-1, M, k, n) # equivalent to 1 - hypergeom.cdf(x-1, M, k, n)
# Convert to numeric
pvals = pvals.astype(float)

# -----------------------------
# Multiple testing correction (FDR)
# -----------------------------
pvals_adj = pd.DataFrame(
    multipletests(pvals.values.flatten(), method='fdr_bh')[1].reshape(pvals.shape),
    index=pvals.index,
    columns=pvals.columns
)

# -----------------------------
# -log10 transformation for heatmap
# -----------------------------
log_pvals = -np.log10(pvals_adj + 1e-300)  # small epsilon to avoid log(0)
#log_pvals = np.transpose(log_pvals.loc[:,~log_pvals.columns.str.contains('all_nuclei|pos')])

In [119]:
# Write Results

# Re-order
celltypes = ['all_nuclei','L2.3.IT','L4.5.IT','L5.6.IT.Car3','L5.6.NP',
             'L5.ET','L6.CT','L6.IT','LAMP5','PAX6','PVALB','SST','VIP',
             'Astro','Microglia','Oligo','Opc','Endo']

counts=count_table.T
row_base = counts.index.to_series().str.split('_').str[0]
row_cat = pd.Categorical(row_base, categories=celltypes, ordered=True)
counts = counts.iloc[np.argsort(row_cat),]
pvals=log_pvals.T
pvals = pvals.iloc[np.argsort(row_cat),]

counts.to_csv(DIR+'LQ_Aging_Intersection_counts.csv')
pvals.to_csv(DIR+'LQ_Aging_Intersection_hypergeom_log_pvals.csv')

data=counts.loc[counts.index.str.contains('pos'),:]
data1=pvals.loc[counts.index.str.contains('pos'),:]
data.to_csv(DIR+'LQ_Aging_Intersection_counts_pos.csv')
data1.to_csv(DIR+'LQ_Aging_Intersection_hypergeom_log_pvals_pos.csv')

data=counts.loc[counts.index.str.contains('neg'),:]
data1=pvals.loc[counts.index.str.contains('neg'),:]
data.to_csv(DIR+'LQ_Aging_Intersection_counts_neg.csv')
data1.to_csv(DIR+'LQ_Aging_Intersection_hypergeom_log_pvals_neg.csv')

In [ ]:
# END